**Preprocessing**

In [2]:
import os

# # Limit OS-level process affinity to exclusively use Core 0.
# # This strictly physically constraints Polars, PyTorch, and everything else in this notebook to 1 core.
os.sched_setaffinity(0, {1, 2, 3})

# Also keep your GPU and general thread limits just in case
#os.environ["CUDA_VISIBLE_DEVICES"] = "0" 
os.environ["OMP_NUM_THREADS"] = "3"
os.environ["POLARS_MAX_THREADS"] = "3" # Must be set before 'import polars'

In [3]:
print(os.sched_getaffinity(0))

{1, 2, 3}


In [1]:
#load libs
from preprocessing import NewsPreprocessing
import polars as pl
from datetime import datetime
import os
from dotenv import load_dotenv
import importlib
import tfidf_finbert

In [3]:
#reload finbert
importlib.reload(tfidf_finbert)
from tfidf_finbert import OptimizedFinBERT

In [2]:
#reload tfidf
importlib.reload(tfidf_finbert)
from tfidf_finbert import TfIdfVectorizer

In [5]:
#load preprocessor
files_to_load_tfIdf = "/mnt/windows/windows_hanka_bcthesis/full_news/cleaned_news_external.parquet"
preprocessor = NewsPreprocessing(files_to_load_tfIdf)

Initializing NLTK resources...


**TF-IDF**

In [8]:
#Loading necessary cleaned data for TF-IDF
filepath = "/mnt/windows/windows_hanka_bcthesis/full_news/cleaned_news_nasdaq_chunks/chunk_0000.parquet"
df_cleaned = pl.read_parquet(filepath)

display(df_cleaned)

trading_session_date_utc,daily_text,preprocessed_for_tfidf
date,str,str
1914-12-12,"""1914. Das ist Nesteroff!""","""num da ist nesteroff"""
1969-12-31,"""Montpelier Re Holdings Ltd. (M…","""montpeli hold ltd mrh new anal…"
2006-10-20,"""Inco's Net Soars on Higher Met…","""inco net soar higher metal pri…"
2006-10-23,"""Jim Cramer: Diageo, Anheuser-B…","""jim cramer diageo anheuserbusc…"
2006-10-24,"""Huaneng Power's Third-Quarter …","""huaneng power thirdquart profi…"
…,…,…
2024-01-03,"""3 Stocks to Buy for the $100,0…","""num stock buy numnum portfolio…"
2024-01-04,"""Intel Will Take Another Swing …","""intel take anoth swing nvidia …"
2024-01-05,"""The Semiconductor Sector Rocke…","""semiconductor sector rocket nu…"


In [5]:
Tf_Idf_Vectorizer_Object = TfIdfVectorizer(dataset = df_cleaned)
train_text, test_text, cleaned_dataset = Tf_Idf_Vectorizer_Object.split_data(cleaned_dataset = df_cleaned, train_ratio = 0.7, date_col = "trading_session_date_utc")

Ready (Row-Count based split)!
Total Unique Days (Rows): 4328
Train Ratio: 70.0%
Split Index: 3029
Cutoff Date: 2018-11-08
Train Period: 1914-12-12 to 2018-11-08 (Exclusive)
Train samples: 3029, Test samples: 1299


preprocessed_for_tfidf
str
"""num da ist nesteroff"""
"""montpeli hold ltd mrh new anal…"
"""inco net soar higher metal pri…"
"""jim cramer diageo anheuserbusc…"
"""huaneng power thirdquart profi…"
…
"""aaon aaon miss qnum earn reven…"
"""notabl friday option activ aal…"
"""offici aphria trade nyse buy n…"


In [6]:
df_tfidf_result = Tf_Idf_Vectorizer_Object.fit_transform(cleaned_dataset, train_text, test_text, date_col = "trading_session_date_utc")
df_tfidf_result.write_parquet("/mnt/windows/windows_hanka_bcthesis/full_news/tfidf_nasdaq.parquet")

Initializing TfidfVectorizer (max_features=5000)...
Fitting TF-IDF on training data...
Train Shape: (3029, 5000)
Transforming test data...
Test Shape:  (1299, 5000)


In [7]:
Tf_Idf_Vectorizer_Object.analyse_results(df_tfidf_result)

Final Shape: (4328, 5001)
Combined TF-IDF DataFrame:


shape: (4_328, 5_001)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────


--- TOP 20 WORDS BY IMPORTANCE ---
shape: (20, 2)
┌─────────┬───────────────┐
│ word    ┆ average_tfidf │
│ ---     ┆ ---           │
│ str     ┆ f64           │
╞═════════╪═══════════════╡
│ num     ┆ 0.578679      │
│ stock   ┆ 0.206982      │
│ numnum  ┆ 0.16677       │
│ market  ┆ 0.110543      │
│ earn    ┆ 0.103186      │
│ updat   ┆ 0.100142      │
│ qnum    ┆ 0.097387      │
│ report  ┆ 0.091728      │
│ announc ┆ 0.090521      │
│ etf     ┆ 0.087446      │
│ new     ┆ 0.086389      │
│ buy     ┆ 0.081         │
│ reg     ┆ 0.075764      │
│ share   ┆ 0.06954       │
│ valu    ┆ 0.061824      │
│ asset   ┆ 0.060252      │
│ net     ┆ 0.059463      │
│ result  ┆ 0.057368      │
│ say     ┆ 0.053943      │
│ analyst ┆ 0.048216      │
└─────────┴───────────────┘
SUCCESS: The word 'the' was successfully removed.


**FinBERT applied on a subset of FNSPID financial news**
- we don't need to fine-tune FinBERT on financial news first. For now the pre-trained model is good enough.


In [6]:
#load news for finbert
files_to_load_finBert = "/mnt/windows/windows_hanka_bcthesis/full_news/nasdaq_external_news.parquet"  
finb_news = preprocessor.load_data(file_paths=files_to_load_finBert, start= datetime(2015, 1, 1, 0, 0, 0),
                                   end= datetime(2017, 12, 31, 23, 59, 59),
                col_to_check='Article_title', stock_symbol=None, 
                filter_russian=True)


 Loading data process started...
   Source: /mnt/windows/windows_hanka_bcthesis/full_news/nasdaq_external_news.parquet
   Filtering for stock: None
   Applying start filter: >= 2015-01-01 00:00:00
   Applying end filter: <= 2017-12-31 23:59:59
    Filtering out Russian text in 'Article_title'...
Loaded 2638114 rows into 'source_1'.


In [7]:
#pick just correct cols for finbert
finbert_df = (
    finb_news
    .with_row_index("id")
    .select([
        pl.col("id"),
        pl.col("date_utc").cast(pl.Datetime),
        pl.col("Article_title"),
    ])
    .filter(pl.col("Article_title").is_not_null())
    .sort("date_utc")
)

#with pl.Config(fmt_str_lengths=1000, tbl_width_chars=1000, tbl_rows=100):
display(finbert_df)

id,date_utc,Article_title
u32,datetime[μs],str
16463,2015-01-01 00:00:00,"""What's Your Deduction?"""
73163,2015-01-01 00:00:00,"""4 Bank Stocks You Should Avoid…"
87845,2015-01-01 00:00:00,"""3 Dividend Aristocrats With Po…"
94038,2015-01-01 00:00:00,"""Top Restaurant Stocks to Buy f…"
94039,2015-01-01 00:00:00,"""Shake Shack IPO: How the Shack…"
…,…,…
1476215,2017-12-31 00:00:00,"""2018 Week 1 Breakout Forecast:…"
1476534,2017-12-31 00:00:00,"""China factory growth eases sli…"
1480251,2017-12-31 00:00:00,"""China factory growth eases sli…"


In [10]:
#take smaller subset for short testing with just 50 articles per day just for shorter time frame
finBert_df_sample = (
    finbert_df
    .filter(
        (pl.col("date_utc") >= pl.datetime(2010, 1, 1, 0, 0, 0)) & 
        (pl.col("date_utc") <= pl.datetime(2012, 12, 31, 23, 59, 59))
    )
  #  .group_by("Date")
 #   .head(6)
)

display(finBert_df_sample)

id,date_utc,Article_title
u32,datetime[μs],str
707741,2010-01-01 00:00:00,"""The Hire Powers Speak Out"""
855624,2010-01-01 00:00:00,"""Paul Frank's Recipe for a Succ…"
1670035,2010-01-01 00:00:00,"""The Hire Powers Speak Out"""
1815809,2010-01-01 00:00:00,"""Paul Frank's Recipe for a Succ…"
1930124,2010-01-01 00:00:00,"""Paul Frank's Recipe for a Succ…"
…,…,…
7517917,2012-12-31 23:22:00,"""U.S. House will consider Senat…"
7517916,2012-12-31 23:23:00,"""House will consider Senate's f…"
7517914,2012-12-31 23:42:00,"""LX Ventures Announces Financia…"


In [4]:
#load hugging face token and create finbert object
load_dotenv() 
# 2. Get the token safely
hf_token = os.getenv("HF_TOKEN")
if hf_token is None:
    raise ValueError(" HF_TOKEN not found! Check your .env file.")
print(f"Token loaded successfully: {hf_token[:4]}...")

#create finbert object
#device = torch.device("cpu")
finbert = OptimizedFinBERT(hf_token=hf_token)

Token loaded successfully: hf_C...


/home/sj5/.micromamba/envs/thesis_dl_2/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


🎯 Locking FinBERT strictly to GPU 0 (NVIDIA GeForce RTX 3080)


In [1]:
#delete finbert object and free up GPU memory
# 1. Delete the actual object
if 'finbert' in locals():
    del finbert

# 2. Force Python's Garbage Collector to find the dead object
import gc
gc.collect()

# 3. Tell PyTorch to release the emptied VRAM back to the GPU driver
import torch
torch.cuda.empty_cache()

In [5]:
#apply finbert model methods and save results
output_file = "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2010-2011July_raw.parquet"
#finbert_results_df = finbert.apply_model(finbert_df, output_file = output_file)
# finbert_results_df.write_parquet("/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2006-2010_raw.parquet")

# #result_file = "/mnt/windows/windows_hanka_bcthesis/full_news/finbert_nasdaq_2006-2010_raw.parquet"

# # 1. Calculate Daily Sentiment (Fast & Light)
daily_sentiment = finbert.aggregate_daily_sentiment(output_file)
daily_sentiment.write_parquet("/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2010-2011July_avg_sentiment.parquet")

# # 2. Calculate Daily Embeddings (Heavy - Uses Streaming)
daily_embeddings = finbert.aggregate_daily_embeddings(output_file)
daily_embeddings.write_parquet("/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2010-2011July_avg_embeddings.parquet")

 Aggregating sentiment scores by Date...
 Mapping to NYSE trading sessions...
   Mapping to NYSE trading sessions (Lazy)...
Aggregated into 378 daily rows.


trading_session_date_utc,avg_pos,avg_neg,avg_neu,daily_article_count
date,f32,f32,f32,u32
2010-01-04,0.213581,0.156469,0.62995,3599
2010-01-05,0.2499,0.126261,0.62384,3664
2010-01-06,0.253521,0.117843,0.628635,3578
2010-01-07,0.243472,0.144662,0.611864,3795
2010-01-08,0.223101,0.161083,0.615816,2722
…,…,…,…,…
2011-06-27,0.217384,0.144918,0.637696,6551
2011-06-28,0.229978,0.12664,0.643384,6235
2011-06-29,0.225297,0.131444,0.643258,6240


 Aggregating Embeddings by Date (Mean Pooling via Polars)...
   1. Building lightweight date map (Ignoring embeddings)...
   Mapping to NYSE trading sessions (Lazy)...
   2. Collecting date map to memory...
   Found 378 unique trading_session_date_utcs.
   3. Processing days individually (Fetching by Row ID)...


Daily Aggregation:   0%|          | 0/378 [00:00<?, ?it/s]

   Stitching (trading) days together...
Created (trading) Daily Embeddings: 378 (trading) days.


trading_session_date_utc,daily_embedding
date,list[f64]
2010-01-04,"[0.028929, 0.146524, … 0.032607]"
2010-01-05,"[0.043389, 0.187731, … 0.014396]"
2010-01-06,"[0.035056, 0.186238, … 0.008968]"
2010-01-07,"[0.041038, 0.185023, … 0.021449]"
2010-01-08,"[0.050207, 0.143474, … 0.028581]"
…,…
2011-06-27,"[0.068211, 0.169951, … 0.091476]"
2011-06-28,"[0.071197, 0.183007, … 0.085624]"
2011-06-29,"[0.06919, 0.182465, … 0.09647]"


In [2]:
import os
print(os.getpid())

142136


In [5]:
#stitch the chunks together from interrupted finbert session
import polars as pl
import glob
import os

print("Scanning and stitching backup chunks individually...")

valid_files = []
# Get all files and sort them numerically by the chunk number
chunk_files = sorted(glob.glob("/mnt/red/red_hanka_bcthesis/finbert_backups/backup_chunk_*.parquet"), 
                     key=lambda x: int(x.split('_')[-1].split('.')[0]))

for file in chunk_files:
    try:
        # Try to read the file completely into memory
        pl.read_parquet_schema(file)
        valid_files.append(file)
    except Exception as e:
        # If it crashes (like chunk 55 likely will), just skip it!
        print(f" Skipping {os.path.basename(file)} (Likely corrupted/incomplete from crash)")

# Combine all successfully loaded DataFrames
if valid_files:
    print(f"\nStitching {len(valid_files)} valid chunks out-of-core (low RAM)...")
    output_file = "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2015-2017_raw.parquet"
    
    # scan_parquet creates a LazyFrame, sink_parquet streams it directly to disk
    pl.scan_parquet(valid_files).sink_parquet(output_file)

    print(f" Successfully saved recovered data to: {output_file}")
    # Display just the final row count to verify, without loading the whole thing
    final_count = pl.scan_parquet(output_file).select(pl.len()).collect().item()
    print(f"\nStitched DataFrame Total Rows: {final_count}")

    # Display a preview
    #display(stitched_df)
else:
    print("No valid chunks were found!")

Scanning and stitching backup chunks individually...

Stitching 53 valid chunks out-of-core (low RAM)...
 Successfully saved recovered data to: /mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2015-2017_raw.parquet

Stitched DataFrame Total Rows: 2638114


In [10]:
#stitch together different finbert results files (aka 2010+2011 etc.)
file_groups = [
    # {
    #     "name": "raw",
    #     "files": [
    #         "/mnt/windows/windows_hanka_bcthesis/full_news/finbert_external_1914-2010_raw.parquet",
    #         "/mnt/windows/windows_hanka_bcthesis/full_news/finbert_external_2010-2018_raw.parquet",
    #         "/mnt/windows/windows_hanka_bcthesis/full_news/finbert_external_2018_raw.parquet",
    #         "/mnt/windows/windows_hanka_bcthesis/full_news/finbert_external_2019_raw.parquet",
    #         "/mnt/red/red_hanka_bcthesis/full_news/finbert_external_2020_raw.parquet"


    #     ],
    #     "output": "/mnt/red/red_hanka_bcthesis/full_news/finbert_external_1914-2020_raw.parquet"
    # },
    {
        "name": "avg_sentiment",
        "files": [
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2006-2010_avg_sentiment.parquet",
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2010-2011July_avg_sentiment.parquet",
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2011July-2013_avg_sentiment.parquet",
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2013-2015_avg_sentiment.parquet",
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2015-2018_avg_sentiment.parquet",
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2018-2020_avg_sentiment.parquet",
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2020-2023_avg_sentiment.parquet",
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2023_avg_sentiment.parquet"
        ],
        "output": "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2006-2023_avg_sentiment.parquet"
    },
    {
        "name": "avg_embeddings",
        "files": [
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2006-2010_avg_embeddings.parquet",
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2010-2011July_avg_embeddings.parquet",
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2011July-2013_avg_embeddings.parquet",
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2013-2015_avg_embeddings.parquet",
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2015-2018_avg_embeddings.parquet",
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2018-2020_avg_embeddings.parquet",
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2020-2023_avg_embeddings.parquet",
            "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2023_avg_embeddings.parquet"
        ],
        "output": "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2006-2023_avg_embeddings.parquet"
    }
]

# Loop through each file group and stitch them
for group in file_groups:
    print(f"Stitching {group['name']} files...")
    # scan_parquet reads laziness, sink_parquet streams directly to disk
    pl.scan_parquet(group["files"]).sink_parquet(group["output"])
    print(f"✓ Saved: {group['output']}")

# To safely check the result without crashing again, use scan to get just the head/row count
lazy_stitch = pl.scan_parquet("/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2006-2023_avg_embeddings.parquet")

# Just fetch the row count
total_rows = lazy_stitch.select(pl.len()).collect().item()
print(f"Total rows in avg_embeddings stitch: {total_rows}")

# Just fetch the first 5 rows to verify
display(lazy_stitch.head(15).collect())
display(lazy_stitch.tail(15).collect())

Stitching avg_sentiment files...
✓ Saved: /mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2006-2023_avg_sentiment.parquet
Stitching avg_embeddings files...
✓ Saved: /mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2006-2023_avg_embeddings.parquet
Total rows in avg_embeddings stitch: 4583


trading_session_date_utc,daily_embedding
date,list[f64]
2006-10-20,"[0.483489, -0.294249, … -0.267092]"
2006-10-23,"[-0.078943, 0.128908, … 0.078362]"
2006-10-24,"[-0.083753, 0.157798, … -0.010952]"
2006-10-25,"[-0.002071, -0.131696, … 0.146072]"
2006-10-27,"[0.106593, 0.031849, … 0.04506]"
…,…
2006-11-08,"[-0.082175, 0.264119, … 0.006181]"
2006-11-09,"[0.117458, 0.101488, … 0.00688]"
2006-11-10,"[0.14785, 0.178693, … 0.216431]"


trading_session_date_utc,daily_embedding
date,list[f64]
2023-12-18,"[0.179682, 0.016104, … 0.035162]"
2023-12-19,"[0.220927, 0.163841, … -0.072486]"
2023-12-20,"[0.26502, -0.112327, … -0.050042]"
2023-12-21,"[0.298363, -0.15426, … -0.025088]"
2023-12-22,"[0.122205, 0.119418, … 0.144292]"
…,…
2024-01-03,"[0.1778, 0.024666, … 0.065817]"
2024-01-04,"[0.034107, -0.039114, … -0.109402]"
2024-01-05,"[0.105175, 0.093692, … 0.022849]"


In [8]:
#take subset of main processed finbert_raw and inspect it
from datetime import datetime
import polars as pl

path_1 = "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2015-2017_avg_embeddings.parquet"
path_2 = "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2018-2020_avg_embeddings.parquet"

start = datetime(2017, 12, 1, 0, 0, 0)
end   = datetime(2018, 3, 1, 0, 0, 0)

# Lazy frame: filter time period
lazy_frame_1 = (
    pl.scan_parquet(path_1)
    .filter(
        (pl.col("trading_session_date_utc") >= pl.lit(start)) &
        (pl.col("trading_session_date_utc") <  pl.lit(end))
    )
)
# Lazy frame: filter time period
lazy_frame_2 = (
    pl.scan_parquet(path_2)
    .filter(
        (pl.col("trading_session_date_utc") >= pl.lit(start)) &
        (pl.col("trading_session_date_utc") <  pl.lit(end))
    )
)
#lazy_frame_1 = (pl.scan_parquet(path_1))

# Small preview (still efficient)
# display(lazy_frame.head(20).collect())
# display(lazy_frame.tail(20).collect())
display(lazy_frame_1.collect())
display(lazy_frame_2.collect())

# Optional: write filtered year directly to disk using streaming (no full RAM load)
# lazy_frame.sink_parquet("/mnt/windows/hanka_bcthesis/full_news/finbert_external_2013_raw.parquet")
#print("Lazy_frame rows:", lazy_frame.select(pl.len()).collect().item())

trading_session_date_utc,daily_embedding
date,list[f64]
2017-12-01,"[0.094407, 0.017599, … -0.030583]"
2017-12-04,"[0.104806, 0.03445, … -0.009201]"
2017-12-05,"[0.103451, 0.061734, … -0.002524]"
2017-12-06,"[0.093747, 0.060936, … 0.015316]"
2017-12-07,"[0.093748, 0.044704, … -0.011531]"
…,…
2017-12-26,"[0.096089, 0.045704, … 0.001527]"
2017-12-27,"[0.096008, 0.04174, … 0.004186]"
2017-12-28,"[0.094173, 0.011315, … -0.024672]"


trading_session_date_utc,daily_embedding
date,list[f64]
2018-01-02,"[0.073657, 0.038151, … -0.02271]"
2018-01-03,"[0.123611, 0.051465, … 0.000755]"
2018-01-04,"[0.113897, 0.039845, … -0.014192]"
2018-01-05,"[0.128463, 0.07914, … 0.009796]"
2018-01-08,"[0.093237, 0.063254, … -0.012509]"
…,…
2018-02-22,"[0.037909, 0.039361, … -0.008374]"
2018-02-23,"[0.08713, 0.035638, … -0.02033]"
2018-02-26,"[0.056938, 0.044681, … -0.013694]"


In [ ]:
#cut off end of parquet file
import polars as pl
from datetime import datetime

src = "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2010-2012_raw.parquet"
dst = "/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2010-2011July_raw.parquet"

cutoff = datetime(2011, 6, 30, 23, 59, 59)

(
    pl.scan_parquet(src)
    .filter(pl.col("date_utc") <= pl.lit(cutoff))
    .sink_parquet(dst)
)

# quick sanity check
rows = pl.scan_parquet(dst).select(pl.len()).collect().item()
min_max = pl.scan_parquet(dst).select(
    pl.col("date_utc").min().alias("min_date"),
    pl.col("date_utc").max().alias("max_date")
).collect()

print("Rows:", rows)
print(min_max)

Rows: 1941627
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-01-01 00:00:00 ┆ 2011-06-30 23:57:00 │
└─────────────────────┴─────────────────────┘


In [ ]:
#search for non-midnight times in date_utc column
from datetime import time

# rows in lazy_frame where date_utc time is NOT midnight
non_midnight = (
    lazy_frame
    .filter(pl.col("date_utc").dt.time() != time(0, 0, 0))
)

# preview a few rows only (still lazy until collect)
display(non_midnight.head(20).collect())

# optional: quick count
print("Non-midnight rows:", non_midnight.select(pl.len()).collect().item())
# ...existing code...

NameError: name 'lazy_frame' is not defined

In [ ]:
#create fake_df for testing
from datetime import datetime

# Create a small fake DataFrame with 8 rows
fake_df = pl.DataFrame({
    "date_utc": [
        datetime(2020, 6, 1, 14, 30),   # Monday
        datetime(2020, 6, 1, 9, 15),    # Monday
        datetime(2020, 6, 1, 20, 45),   # Monday
        datetime(2020, 6, 5, 17, 20),   # Friday
        datetime(2020, 6, 6, 10, 0),    # Saturday (weekend)
        datetime(2020, 6, 7, 15, 30),   # Sunday (weekend)
        datetime(2020, 6, 8, 8, 45),    # Monday
        datetime(2020, 6, 9, 21, 10),   # Tuesday
    ],
    "Article_title": [
        "Tech stocks rally amid optimism",
        "Federal Reserve holds rates steady",
        "Oil prices drop on demand concerns",
        "Unemployment claims fall unexpectedly",
        "Markets closed for weekend trading",
        "Analysts predict strong earnings season",
        "New trade deal announced with Europe",
        "Inflation data exceeds expectations",
    ],
    "Stock_symbol": ["AAPL"] * 8  # Optional: if you need this column
})

# Add UTC timezone
fake_df = fake_df.with_columns(
    pl.col("date_utc").dt.replace_time_zone("UTC")
)

display(fake_df.sort("date_utc"))

date_utc,Article_title,Stock_symbol
"datetime[μs, UTC]",str,str
2020-06-01 09:15:00 UTC,"""Federal Reserve holds rates st…","""AAPL"""
2020-06-01 14:30:00 UTC,"""Tech stocks rally amid optimis…","""AAPL"""
2020-06-01 20:45:00 UTC,"""Oil prices drop on demand conc…","""AAPL"""
2020-06-05 17:20:00 UTC,"""Unemployment claims fall unexp…","""AAPL"""
2020-06-06 10:00:00 UTC,"""Markets closed for weekend tra…","""AAPL"""
2020-06-07 15:30:00 UTC,"""Analysts predict strong earnin…","""AAPL"""
2020-06-08 08:45:00 UTC,"""New trade deal announced with …","""AAPL"""
2020-06-09 21:10:00 UTC,"""Inflation data exceeds expecta…","""AAPL"""
